In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("./GA_2_dataset.csv")
data.head()

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,35900,37.0,Male,Other,Strategy,23.929404,NaN,Hard,3,124,99,18,Medium
1,27085,25.0,Male,NaN,Action,22.755168,1.0,Easy,14,84,84,12,Medium
2,39595,24.0,Female,Europe,Simulation,19.505292,0.0,Hard,3,172,9,18,Medium
3,37440,26.0,Female,Europe,RPG,11.009645,NaN,NaN,3,83,36,43,Low
4,22882,17.0,Female,USA,RPG,0.581039,1.0,Medium,5,163,9,24,Medium


Q1: Which of the following columns have object datatype?

In [3]:
# Approach 1
data.select_dtypes(include='object').columns

Index(['Gender', 'Location', 'GameGenre', 'GameDifficulty', 'EngagementLevel'], dtype='object')

In [4]:
# Approach 2
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PlayerID                   10000 non-null  int64  
 1   Age                        9200 non-null   float64
 2   Gender                     10000 non-null  object 
 3   Location                   9202 non-null   object 
 4   GameGenre                  10000 non-null  object 
 5   PlayTimeHours              10000 non-null  float64
 6   InGamePurchases            9107 non-null   float64
 7   GameDifficulty             9154 non-null   object 
 8   SessionsPerWeek            10000 non-null  int64  
 9   AvgSessionDurationMinutes  10000 non-null  int64  
 10  PlayerLevel                10000 non-null  int64  
 11  AchievementsUnlocked       10000 non-null  int64  
 12  EngagementLevel            10000 non-null  object 
dtypes: float64(3), int64(5), object(5)
memory usage

In [14]:
data.isnull().sum() # Age  Location  InGamePurchases GameDifficulty  --> have null values                                                        

PlayerID                       0
Age                          800
Gender                         0
Location                     798
GameGenre                      0
PlayTimeHours                  0
InGamePurchases              893
GameDifficulty               846
SessionsPerWeek                0
AvgSessionDurationMinutes      0
PlayerLevel                    0
AchievementsUnlocked           0
EngagementLevel                0
dtype: int64

Q2: In this dataset, how many "Males" from "Europe" have made "InGamePurchases" ?

In [5]:
data[(data["Gender"] == "Male") & (data['Location'] == "Europe") & (data["InGamePurchases"] == 1.0)].shape

(299, 13)

Q3: In your dataset, how many players under the "Age" 18 have strictly greater than 10 "PlayTimeHours"?


In [6]:
data[(data["Age"] < 18) & (data["PlayTimeHours"] > 10)].shape

(453, 13)

Q4: Create feature matrix(X) and label vector(y) using following instructions:
- "EngagementLevel" is the target column(y).
- All the columns except the target column are in feature matrix(X).

How many total null values were present in the whole dataset ?

In [8]:
y = data["EngagementLevel"]
X = data.drop(columns=["EngagementLevel"],axis=1)

data.isna().sum().sum()

3337

Q5: Split the dataset into train dataset and test dataset in the following manner.
- Use sklearn train_test_split function to split the data.
- Use only 20% data as test_set and keep random_state = 42

Which category has the least value counts in y_train?

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

y_train.value_counts()

EngagementLevel
Medium    3983
Low       2021
High      1996
Name: count, dtype: int64

Refer to the questions on the portal

Q6: Write the sum of transformed(imputed) "Age" column of the test dataset . (upto 2 digits after decimal points)

Rules for Imputing the missing(NaN) or Unknown values:

- Calculating statistical values (such as mean, median, mode) for each column in the training dataset.
- Applying these calculated statistical values to replace missing (NaN) and unknown values in both the training and test datasets.
- Ensure that the calculation of statistical values excludes any rows containing missing or unknown values.
- Replace Unknown values in the "Age" feature with the Mean value in that.
- Replace Unknown values in the "Location" feature with the constant value "Other".
- Replace Unknown values in the "GameDifficulty" feature with the Most Frequent value in that.
- Replace Unknown values in the "InGamePurchases" feature with the constant value '0'.
- Write the answers related to the above imputation in below questions respectively.

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

transformer_list = [
    ("mean_imp_Transformer",SimpleImputer(strategy="mean"),["Age"]),
    ("const_imp_Transformer",SimpleImputer(strategy="constant",fill_value="Other"),["Location"]),
    ("mode_imp_Transformer",SimpleImputer(strategy="most_frequent"),["GameDifficulty"]),
    ("const_2_imp_Transformer",SimpleImputer(strategy="constant",fill_value=0),["InGamePurchases"])

]

ct = ColumnTransformer(transformers = transformer_list, remainder='passthrough',verbose_feature_names_out=False).set_output(transform="pandas")
ct

ColumnTransformer(remainder='passthrough',
                  transformers=[('mean_imp_Transformer', SimpleImputer(),
                                 ['Age']),
                                ('const_imp_Transformer',
                                 SimpleImputer(fill_value='Other',
                                               strategy='constant'),
                                 ['Location']),
                                ('mode_imp_Transformer',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['GameDifficulty']),
                                ('const_2_imp_Transformer',
                                 SimpleImputer(fill_value=0,
                                               strategy='constant'),
                                 ['InGamePurchases'])],
                  verbose_feature_names_out=False)

In [15]:
X_train_impute = ct.fit_transform(X_train)
X_test_impute = ct.transform(X_test)

In [22]:
X_train_impute.head(2)

,Age,Location,GameDifficulty,InGamePurchases,PlayerID,Gender,GameGenre,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked
9254,34.0,Asia,Medium,0.0,43413,Female,Simulation,15.777862,5,76,80,33
1561,41.0,USA,Medium,0.0,43503,Male,Strategy,3.044182,7,73,34,5


In [20]:
X_train_impute.isnull().sum()

Age                          0
Location                     0
GameDifficulty               0
InGamePurchases              0
PlayerID                     0
Gender                       0
GameGenre                    0
PlayTimeHours                0
SessionsPerWeek              0
AvgSessionDurationMinutes    0
PlayerLevel                  0
AchievementsUnlocked         0
dtype: int64

In [23]:
X_test_impute.head(2)

,Age,Location,GameDifficulty,InGamePurchases,PlayerID,Gender,GameGenre,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked
6252,38.0,Other,Easy,0.0,45359,Male,Strategy,13.585655,18,141,96,19
4684,37.0,Europe,Easy,0.0,36195,Female,Simulation,3.650090,17,12,22,2


In [21]:
X_test_impute.isna().sum()

Age                          0
Location                     0
GameDifficulty               0
InGamePurchases              0
PlayerID                     0
Gender                       0
GameGenre                    0
PlayTimeHours                0
SessionsPerWeek              0
AvgSessionDurationMinutes    0
PlayerLevel                  0
AchievementsUnlocked         0
dtype: int64

In [24]:
X_test_impute["Age"].sum() ## answer

63585.239130434784

Q7: Calculate the sum of all the values present in first five rows of transformed test feature matrix ? (upto 2 digits afer the decimal)

In [25]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [26]:
X_train_impute.drop('PlayerID', axis=1, inplace=True)
X_test_impute.drop('PlayerID', axis=1, inplace=True)

In [33]:
trf = [
    ("OHE_Transformer",OneHotEncoder(sparse_output=False ,drop = 'first'),['Gender', 'Location', 'GameGenre']),
    ("OE_Transformer",OrdinalEncoder(categories=[['Easy','Medium','Hard']]),["GameDifficulty"])
]

ct2 = ColumnTransformer(trf, remainder='passthrough',verbose_feature_names_out=False).set_output(transform="pandas")

ct2

ColumnTransformer(remainder='passthrough',
                  transformers=[('OHE_Transformer',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['Gender', 'Location', 'GameGenre']),
                                ('OE_Transformer',
                                 OrdinalEncoder(categories=[['Easy', 'Medium',
                                                             'Hard']]),
                                 ['GameDifficulty'])],
                  verbose_feature_names_out=False)

In [28]:
X_train_encode = ct2.fit_transform(X_train_impute)
X_test_encode = ct2.transform(X_test_impute)

In [32]:
X_train_encode.head(2)


,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy,GameDifficulty,Age,InGamePurchases,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked
9254,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,34.0,0.0,15.777862,5,76,80,33
1561,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,41.0,0.0,3.044182,7,73,34,5


In [34]:
X_test_encode.head(2)

,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy,GameDifficulty,Age,InGamePurchases,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked
6252,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,38.0,0.0,13.585655,18,141,96,19
4684,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,37.0,0.0,3.650090,17,12,22,2


In [29]:
ss = StandardScaler().set_output(transform="pandas")
X_train_ss = ss.fit_transform(X_train_encode)

X_test_ss = ss.transform(X_test_encode)

In [30]:
X_test_ss.iloc[:5,:].values.sum()

-7.172844117420558